In [ ]:
import pandas as pd
import geopandas as geopandas
import hkvsobekpy
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [1, 2]

scenarios = ["REF", "SCEN"]

start_date = "2010-4-1"
# end_date = "2018-12-31"
end_date = "2010-4-12"

# path to the package containing the dummy-data
dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\"

In [ ]:
simulaties_total = pd.DataFrame()

for selectie_gebied in selectie_gebieden:
    for scenario in scenarios:
        # date_range = pd.date_range(start_date, end_date, freq="6MS")
        date_range = pd.date_range(start_date, end_date, freq="2D")

        simulaties = pd.DataFrame()
        simulaties["start_date"] = date_range
        simulaties["end_date"] = simulaties["start_date"].shift(-1)
        simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
        simulaties["seizoen"] = ["zomer", "winter"]*int(len(date_range)/2)
        simulaties["scenario"] = scenario
        simulaties["gebied"] = selectie_gebied
        simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])

        simulaties["model_name"] = simulaties.apply(lambda x: f"rr_model__{pd.to_datetime(x.start_date).strftime('%Y_%m_%d')}__{pd.to_datetime(x.end_date).strftime('%Y_%m_%d')}", axis=1)
        simulaties_total = pd.concat([simulaties_total, simulaties])

In [ ]:
total_link_flows = pd.DataFrame()

for selectie_gebied in selectie_gebieden:
    for scenario in scenarios:
        simulaties = simulaties_total[(simulaties_total["gebied"]==selectie_gebied) & (simulaties_total["scenario"]==scenario)]
        link_flows = pd.DataFrame()
        for index, simulatie in simulaties.iterrows():
            print(str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)

            dir_model = Path(dir_model_basis, f"gebied_{simulatie.gebied}", simulatie.scenario)
            unpaved_rr_file = "3blinks.his"

            path_unpaved_rr_file = Path(dir_model, simulatie.model_name, "rr", unpaved_rr_file)
            if not path_unpaved_rr_file.exists():
                print(f"File {path_unpaved_rr_file} does not exist. Skipping.")
                continue
            rr_his = hkvsobekpy.read_his.ReadMetadata(path_unpaved_rr_file)
            rr_results_link_flow = rr_his.DataFrame()['Link flow [m3/s]    ']
            link_flows = pd.concat([link_flows, rr_results_link_flow])
        
        total_link_flows[f"{selectie_gebied}_{scenario}"] = link_flows.sum(axis=1)        

In [ ]:
total_link_flows

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10,6))
total_link_flows.plot(ax=ax)
ymin = 0
ymax = total_link_flows.max().max()*1.1
ax.vlines(simulaties.start_date, ymin=ymin, ymax=ymax, color="lightgrey", linestyles="dashed")
ax.set_ylim([ymin, ymax]);